In [1]:
# Download data locally
import os
import re

import pandas as pd
import xarray as xr
from dotenv import find_dotenv, load_dotenv

In [2]:
load_dotenv(find_dotenv("november_gs.env", raise_error_if_not_found=True), override=True)

EFFECTS_URI = os.environ["POREALLAS_EFFECTS_URI"]
IMPACT_REGION_POLYGONS = os.environ["POREALLAS_REGIONS_POLYGONS_URI"]
SOCIOECONOMICS_URI = os.environ["POREALLAS_SOCIOECONOMICS_URI"]

# Climate Data
TAS_FORECAST_ADJ = os.environ["POREALLAS_TAS_FORECAST_URI"]
ERA5_ADJ = os.environ["POREALLAS_ERA5_URI"]
TAS_FORECAST_RAW = os.environ["POREALLAS_PARSED_FORECAST_URI"]

In [3]:
RUN_MONTH = "2026-11"
N_LEADS = 6

_months = pd.period_range(pd.Period(RUN_MONTH, "M"), periods=N_LEADS, freq="M")
FC_MONTHS = [p.month for p in _months]
FC_PERIOD = slice(
    _months[0].start_time.strftime("%Y-%m-%d"),
    _months[-1].end_time.strftime("%Y-%m-%d"),
)

# Baseline period for Impact
BASELINE_PERIOD = slice("1996-01-01", "2025-12-31")

print("OUTPUT PARAMETERS")
print("Forecast Months:", FC_MONTHS)
print(f"Baseline Period: {BASELINE_PERIOD.start} to {BASELINE_PERIOD.stop}")

OUTPUT PARAMETERS
Forecast Months: [11, 12, 1, 2, 3, 4]
Baseline Period: 1996-01-01 to 2025-12-31


In [4]:
def retrieve_version(uri):
    # Automatically set version_date
    m = re.search(r"/v(\d{8})/", uri)
    return f"v{m.group(1)}"

In [5]:
uri = TAS_FORECAST_RAW
version = retrieve_version(uri)

In [6]:
uri = TAS_FORECAST_RAW
version = retrieve_version(uri)
_ds = xr.open_dataset(
    uri,
    engine="zarr",
    chunks={"latitude": 30, "longitude": 30, "time": -1},
    backend_kwargs={"storage_options": {"token": "anon"}},
)

In [ ]:
# Break out forecast into hindcast climatology and ensemble forecast
# Forecast
_ds_fc = _ds.sel(time=FC_PERIOD)
_ds_fc.to_zarr(f"{version}_parsed_forecast.zarr")

In [7]:
# Hindcast
_ds_hc = _ds.sel(time=BASELINE_PERIOD).mean(dim="number")
_ds_hc.to_zarr(f"{version}_parsed_hindcast.zarr")

/home/emily_zuetell/projects/poreallas/.venv/lib/python3.14/site-packages/zarr/api/asynchronous.py:246: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


In [ ]:
uri = EFFECTS_URI
version = retrieve_version(uri)
_ds = xr.open_datatree(
    uri, engine="zarr", chunks={}, backend_kwargs={"storage_options": {"token": "anon"}}
)
_ds.to_zarr(f"../data/v2611/{version}_effects.zarr")

/home/emily_zuetell/projects/poreallas/.venv/lib/python3.14/site-packages/zarr/api/asynchronous.py:231: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


KeyboardInterrupt: 

In [6]:
uri

'gs://poreallas-public-20260605/v20261108/parsed/effects.zarr'